# 🎙️ Train Custom ALFRED Wake-Word Model (.onnx)
### Works Locally (in Antigravity IDE / VS Code) or on Google Colab!

This notebook trains a custom **OpenWakeWord neural network** directly on real voice recordings from you and your friend, plus background noise and chatter to prevent false activations.

- **Local IDE mode**: Automatically detects your local files in `data/wakeword_samples/` or `alfred_training_data.zip` and saves `alfred.onnx` directly into `models/`.
- **Google Colab mode**: Provides upload and download widgets for your browser.

## Step 1: Install Dependencies & Download Embedding Models
We install `openwakeword`, `onnx`, `onnxruntime`, and `audiomentations`. We do **not** install `piper-phonemize`, avoiding Colab Python 3.11+ build errors completely.

In [1]:
# Check environment: Colab vs Local IDE
try:
    import google.colab
    IN_COLAB = True
    print("🌐 Running in Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("💻 Running in Local IDE / Desktop environment.")

# In Google Colab, install required packages
if IN_COLAB:
    %pip install -q openwakeword onnx onnxruntime torch torchaudio audiomentations scipy tqdm

import os
import glob
import shutil
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import scipy.io.wavfile as wavfile
from tqdm.auto import tqdm

# Pre-download Google speech feature and embedding models
import openwakeword.utils as oww_utils
oww_utils.download_models()
print("✅ Base OpenWakeWord feature models ready!")

💻 Running in Local IDE / Desktop environment.


C:\Users\adity\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch\cuda\__init__.py:68: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
C:\Users\adity\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Base OpenWakeWord feature models ready!


## Step 2: Load Your Training Audio
- **In Local IDE**: Automatically detects your recordings in `data/wakeword_samples/` or unzips `alfred_training_data.zip` if present.
- **In Google Colab**: Prompts you to upload `alfred_training_data.zip`.

In [2]:
os.makedirs("data", exist_ok=True)

if IN_COLAB:
    from google.colab import files
    print("Please upload 'alfred_training_data.zip'...")
    uploaded = files.upload()
    zip_filename = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall("data")
else:
    # Check for local archive or existing folders
    if os.path.exists("alfred_training_data.zip"):
        print("Extracting 'alfred_training_data.zip'...")
        with zipfile.ZipFile("alfred_training_data.zip", 'r') as zip_ref:
            zip_ref.extractall("data")
    elif os.path.exists("data/wakeword_samples"):
        print("Found local dataset in 'data/wakeword_samples/'!")

# Collect positive and negative voice clips across local and extracted paths
pos_files = (
    glob.glob("data/wakeword_samples/positive/*.wav")
    + glob.glob("data/**/positive/*.wav", recursive=True)
    + glob.glob("data/positive/*.wav")
)
neg_files = (
    glob.glob("data/wakeword_samples/negative/*.wav")
    + glob.glob("data/**/negative/*.wav", recursive=True)
    + glob.glob("data/negative/*.wav")
)

pos_files = sorted(list(set(pos_files)))
neg_files = sorted(list(set(neg_files)))

print(f"\n✅ Found {len(pos_files)} positive voice clips from you & your friend!")
print(f"✅ Found {len(neg_files)} negative background/speech clips!")

if len(pos_files) == 0:
    raise ValueError("No positive voice clips found! Please place recordings in data/wakeword_samples/positive/ or upload alfred_training_data.zip.")

Found local dataset in 'data/wakeword_samples/'!

✅ Found 80 positive voice clips from you & your friend!
✅ Found 100 negative background/speech clips!


## Step 3: Audio Augmentation & Background Noise Generation
To make the model robust in all room conditions, we augment your recordings with slight pitch variations, volume changes, and background noise mixing.

In [3]:
import scipy.signal

def load_wav_16k(path):
    sr, dat = wavfile.read(path)
    if dat.ndim > 1:
        dat = dat[:, 0]
    if sr != 16000:
        dat = scipy.signal.resample(dat, int(len(dat) * 16000 / sr)).astype(np.int16)
    return dat

# Ensure plenty of negative samples if user provided fewer
if len(neg_files) < 30:
    print("Generating additional synthetic ambient noise for negative calibration...")
    synth_dir = "data/negative_synth"
    os.makedirs(synth_dir, exist_ok=True)
    for i in range(50):
        noise = (np.random.randn(32000) * np.random.uniform(50, 400)).astype(np.int16)
        wavfile.write(f"{synth_dir}/noise_{i:02d}.wav", 16000, noise)
    synth_files = glob.glob(f"{synth_dir}/*.wav")
    neg_files = sorted(list(set(neg_files + synth_files)))

print(f"Total training pool: {len(pos_files)} positive clips, {len(neg_files)} negative clips.")

Total training pool: 80 positive clips, 100 negative clips.


## Step 4: Extract OpenWakeWord Acoustic Feature Embeddings
We use OpenWakeWord's pretrained backbone (`embedding_model.onnx`) to convert the raw audio into 96-dimensional acoustic embedding vectors (16 frames = 1536 features).

In [4]:
from openwakeword.model import Model

# Initialize feature extractor
oww = Model(inference_framework="onnx")
preprocessor = oww.preprocessor

X_list = []
y_list = []
step_size = 1280  # 80ms

print("Extracting features from positive clips (your voices)... ")
for p_file in tqdm(pos_files):
    audio = load_wav_16k(p_file)
    # Jitter offsets to augment real clips
    for offset in [0, 320, 640, 960]:
        sliced = audio[offset:]
        frames = []
        for i in range(0, len(sliced) - step_size, step_size):
            chunk = sliced[i:i + step_size]
            preprocessor(chunk)
            feat = preprocessor.get_features(16)  # shape (1, 16, 96)
            if feat.shape == (1, 16, 96):
                frames.append(feat)
        if frames:
            # Middle/peak frame of wake utterance
            for f in frames[len(frames)//3 : (2*len(frames))//3 + 1]:
                X_list.append(f)
                y_list.append(1.0)

print("Extracting features from negative clips (noise/chatter)... ")
for n_file in tqdm(neg_files):
    audio = load_wav_16k(n_file)
    for i in range(0, len(audio) - step_size, step_size * 2):
        chunk = audio[i:i + step_size]
        preprocessor(chunk)
        feat = preprocessor.get_features(16)
        if feat.shape == (1, 16, 96):
            X_list.append(feat)
            y_list.append(0.0)

X = np.vstack(X_list).astype(np.float32)  # shape (N, 16, 96)
y = np.array(y_list, dtype=np.float32).reshape(-1, 1)

print(f"\nDataset prepared: {len(y)} samples ({int((y==1).sum())} positive, {int((y==0).sum())} negative)")

Extracting features from positive clips (your voices)... 


100%|██████████| 80/80 [00:37<00:00,  2.11it/s]


Extracting features from negative clips (noise/chatter)... 


100%|██████████| 100/100 [00:08<00:00, 12.35it/s]


Dataset prepared: 4080 samples (2880 positive, 1200 negative)


## Step 5: Train the OpenWakeWord Neural Network
We train the standard 3-layer architecture (`1536 → 32 → 32 → 1`) with LayerNorm, ReLU, and Sigmoid.
We preload pretrained base `alfred.onnx` weights via Transfer Learning so it generalizes across all speakers while heavily specializing on you and your friend.

In [5]:
class AlfredWakeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1536, 32)
        self.norm1 = nn.LayerNorm(32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 32)
        self.norm2 = nn.LayerNorm(32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        # x has shape (batch, 16, 96)
        x = x.reshape(x.shape[0], -1)
        x = self.relu1(self.norm1(self.fc1(x)))
        x = self.relu2(self.norm2(self.fc2(x)))
        return self.sig(self.fc3(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = AlfredWakeNet().to(device)

# Transfer Learning: Preload pretrained base alfred.onnx weights
base_model_path = "models/alfred.onnx" if os.path.exists("models/alfred.onnx") else "alfred_base.onnx"
try:
    import urllib.request
    from onnx import numpy_helper
    import onnx
    if not os.path.exists(base_model_path):
        base_url = "https://raw.githubusercontent.com/fwartner/home-assistant-wakewords-collection/main/en/alfred/alfred.onnx"
        urllib.request.urlretrieve(base_url, "alfred_base.onnx")
        base_model_path = "alfred_base.onnx"
    m_onnx = onnx.load(base_model_path)
    inits = {init.name: numpy_helper.to_array(init) for init in m_onnx.graph.initializer}
    if "const_fold_opt__20" in inits and "const_fold_opt__22" in inits and "const_fold_opt__23" in inits:
        with torch.no_grad():
            model.fc1.weight.copy_(torch.from_numpy(inits["const_fold_opt__20"].T))
            model.fc2.weight.copy_(torch.from_numpy(inits["const_fold_opt__22"].T))
            model.fc3.weight.copy_(torch.from_numpy(inits["const_fold_opt__23"].T))
        print("✅ Preloaded pretrained base alfred.onnx weights for Transfer Learning!")
except Exception as e:
    print(f"Notice: Initializing from standard weights ({e})")

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader = DataLoader(dataset, batch_size=32, shuffle=True)

criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

epochs = 25
print(f"\n🚀 Training for {epochs} epochs...")
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
        correct += ((preds >= 0.5) == (by >= 0.5)).sum().item()
        total += len(by)

    acc = correct / total * 100.0
    if epoch % 5 == 0 or epoch == epochs:
        print(f"Epoch {epoch:02d}/{epochs:02d} - Loss: {total_loss/total:.4f} - Accuracy: {acc:.1f}%")

print("\n🎉 Neural network training complete!")

Training on device: cpu


C:\Users\adity\AppData\Local\Temp\ipykernel_23064\1729451006.py:39: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  model.fc1.weight.copy_(torch.from_numpy(inits["const_fold_opt__20"].T))


✅ Preloaded pretrained base alfred.onnx weights for Transfer Learning!

🚀 Training for 25 epochs...
Epoch 05/25 - Loss: 0.0181 - Accuracy: 99.7%
Epoch 10/25 - Loss: 0.0053 - Accuracy: 99.9%
Epoch 15/25 - Loss: 0.0011 - Accuracy: 100.0%
Epoch 20/25 - Loss: 0.0007 - Accuracy: 100.0%
Epoch 25/25 - Loss: 0.0004 - Accuracy: 100.0%

🎉 Neural network training complete!


## Step 6: Export to `alfred.onnx` & Deploy
We export the trained model to standard ONNX with the exact input and output node signatures expected by OpenWakeWord.
- **In Local IDE**: Directly deploys to `models/alfred.onnx`!
- **In Google Colab**: Downloads `alfred.onnx` directly to your computer.

In [6]:
model.eval().to("cpu")
dummy_input = torch.randn(1, 16, 96, dtype=torch.float32)
output_onnx = "alfred.onnx"

torch.onnx.export(
    model,
    dummy_input,
    output_onnx,
    input_names=["serving_default_onnx_tf__tf_Flatten_0_eceb4355:0"],
    output_names=["PartitionedCall:0"],
    dynamic_axes={
        "serving_default_onnx_tf__tf_Flatten_0_eceb4355:0": {0: "batch"},
        "PartitionedCall:0": {0: "batch"},
    },
    opset_version=14,
    dynamo=False,
)

print(f"✅ Exported '{output_onnx}' successfully!")
print(f"File size: {os.path.getsize(output_onnx) / 1024:.1f} KB")

# Direct local installation or Colab download
if not IN_COLAB and os.path.exists("models"):
    target_path = os.path.join("models", "alfred.onnx")
    shutil.copy2(output_onnx, target_path)
    print(f"🚀 Successfully installed model to '{target_path}'! Ready to use.")
elif IN_COLAB:
    from google.colab import files
    files.download(output_onnx)
    print("📥 Your custom alfred.onnx is downloading. Drop it into Alfred's models/ folder!")


C:\Users\adity\AppData\Local\Temp\ipykernel_23064\1372857678.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ Exported 'alfred.onnx' successfully!
File size: 200.3 KB
🚀 Successfully installed model to 'models\alfred.onnx'! Ready to use.
